## Example 1: Parse a URI into its components

In [ ]:
import rfc3987

# Example 1: Parse a URI into its components
print("Example 1: Parse a URI into its components")

try:
    # rfc3987 uses the third-party 'regex' module when it is installed, and the standard 're' otherwise.
    # With 're', parse() returns only the five main components, so only those are checked here.
    print(f"  regex module in use: {rfc3987.REGEX}")
    parts = rfc3987.parse('http://user@example.com:8080/p/q?x=1#frag', rule='URI')
    main = {k: parts[k] for k in ('scheme', 'authority', 'path', 'query', 'fragment')}
    print(f"  {main}")
    assert main == {'scheme': 'http', 'authority': 'user@example.com:8080', 'path': '/p/q',
                    'query': 'x=1', 'fragment': 'frag'}, parts

    # components that are absent come back as None
    parts = rfc3987.parse('mailto:a@example.com', rule='URI')
    assert (parts['scheme'], parts['authority'], parts['path'], parts['query']) == ('mailto', None, 'a@example.com', None), parts

    print("rfc3987 example 1 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3987 example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 finished")

## Example 2: IRIs versus URIs

In [ ]:
import rfc3987

# Example 2: IRIs versus URIs
print("Example 2: IRIs versus URIs")

try:
    iri = 'http://例え.jp/パス?q=値'
    # An IRI may contain non-ASCII characters; a URI may not
    assert rfc3987.match(iri, rule='IRI') is not None
    assert rfc3987.match(iri, rule='URI') is None
    parts = rfc3987.parse(iri, rule='IRI')
    print(f"  authority={parts['authority']!r} path={parts['path']!r} query={parts['query']!r}")
    assert (parts['authority'], parts['path'], parts['query']) == ('例え.jp', '/パス', 'q=値'), parts

    # match() returns a match object or None
    assert rfc3987.match('http://example.com', rule='URI') is not None
    assert rfc3987.match('not a uri', rule='URI') is None
    assert rfc3987.match('../d?x', rule='relative_ref') is not None
    assert rfc3987.match('../d?x', rule='absolute_URI') is None

    print("rfc3987 example 2 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3987 example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 finished")

## Example 3: Resolve relative references against a base URI

In [ ]:
import rfc3987

# Example 3: Resolve relative references against a base URI
print("Example 3: Resolve relative references against a base URI")

try:
    # Normal examples from RFC 3986 section 5.4.1, base 'http://a/b/c/d;p?q'
    base = 'http://a/b/c/d;p?q'
    expected = {
        'g': 'http://a/b/c/g',
        './g': 'http://a/b/c/g',
        'g/': 'http://a/b/c/g/',
        '/g': 'http://a/g',
        '//g': 'http://g',
        '?y': 'http://a/b/c/d;p?y',
        'g?y#s': 'http://a/b/c/g?y#s',
        '..': 'http://a/b/',
        '../g': 'http://a/b/g',
        '../../g': 'http://a/g',
    }
    for ref, target in expected.items():
        resolved = rfc3987.resolve(base, ref)
        print(f"  {ref!r:10} -> {resolved}")
        assert resolved == target, (ref, resolved)

    # return_parts=True gives the components instead of a string
    parts = rfc3987.resolve(base, '../g', return_parts=True)
    assert (parts['scheme'], parts['authority'], parts['path']) == ('http', 'a', '/b/g'), parts

    print("rfc3987 example 3 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3987 example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 finished")

## Example 4: Compose a URI from parts

In [ ]:
import rfc3987

# Example 4: Compose a URI from parts
print("Example 4: Compose a URI from parts")

try:
    uri = rfc3987.compose(scheme='https', authority='example.com', path='/a', query='b=1', fragment='c')
    print(f"  composed: {uri}")
    assert uri == 'https://example.com/a?b=1#c', uri

    # parse() followed by compose() gives back the original string
    for original in ['https://example.com/a/b?q#f', 'urn:isbn:0451450523', 'http://a/b/c/d;p?q']:
        rebuilt = rfc3987.compose(**rfc3987.parse(original, rule='URI'))
        print(f"  {original!r} -> {rebuilt!r}")
        assert rebuilt == original, (original, rebuilt)

    print("rfc3987 example 4 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3987 example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 finished")

## Example 5: Invalid input and non-validating parse

In [ ]:
import rfc3987

# Example 5: Invalid input and non-validating parse
print("Example 5: Invalid input and non-validating parse")

try:
    # A validating parse raises ValueError for text that does not match the rule
    for bad in ['not a uri', 'http://exa mple.com', 'http://example.com/%zz']:
        try:
            rfc3987.parse(bad, rule='URI')
        except ValueError as err:
            print(f"  rejected: {err}")
        else:
            raise AssertionError(f"{bad!r} was parsed as a valid URI")

    # rule=None splits the string with the RFC 3986 appendix B regex, without validating it
    loose = rfc3987.parse('not a uri?x#y', rule=None)
    print(f"  non-validating parse: {loose}")
    assert (loose['scheme'], loose['path'], loose['query'], loose['fragment']) == (None, 'not a uri', 'x', 'y'), loose

    print("rfc3987 example 5 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3987 example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 finished")